### Sanity Checks

#### Events Sub 01

In [1]:
import pandas as pd
from pathlib import Path

tsv_dir = Path('../MNE-sample-data/ds006761/sub-01')
tsv_file = next(tsv_dir.glob('**/*.tsv'), None)

print(tsv_file)

df = pd.read_csv(tsv_file, sep='\t')
df.head()

..\MNE-sample-data\ds006761\sub-01\eeg\sub-01_task-RPS_events.tsv


,onset,duration,onset_sample,trial_num,player1_resp,player1_rt,player2_resp,player2_rt,outcome
0,32.777344,5,67128,1,3,0.558,2,0.516,2
1,37.769043,5,77351,2,3,0.435,3,0.560,1
2,42.760742,5,87574,3,2,0.643,2,0.451,1
3,47.760254,5,97813,4,2,0.717,2,0.459,1
4,52.760254,5,108053,5,1,0.642,3,0.359,2


In [ ]:
import mne
import numpy as np
from pathlib import Path
import sys

# Pipeline config importieren
PIPELINE_PATH = Path("../eeg_pipeline")
if PIPELINE_PATH not in sys.path:
    sys.path.insert(0, str(PIPELINE_PATH))
import config

# --- Lade die Daten ---
SUBJECT = config.SUBJECTS[0]  # z.B. "01"
PERSON = "P1"                  # "P1" oder "P2"

raw_path = config.OUTPUT_DIR / f"sub-{SUBJECT}_{PERSON}_raw.fif"
print(f"Loading: {raw_path}")

raw = mne.io.read_raw_fif(raw_path, preload=True)
print("✅ Raw data loaded.\n")

# Nur EEG-Kanäle auswählen
raw_eeg = raw.copy().pick_types(eeg=True)

# Präfix entfernen (z.B. "1-A1" -> "A1")
prefix = "1-"
mapping_prefix = {ch: ch[len(prefix):] for ch in raw_eeg.ch_names if ch.startswith(prefix)}
raw_eeg.rename_channels(mapping_prefix)

# -----------------------------------------
# 🔥 BioSemi64 → Standard-10–20 Mapping
# -----------------------------------------


bios64_to_1020 = {
    # A row
    "A1": "Fp1", "A2": "AF7", "A3": "AF3", "A4": "F1", "A5": "F3",
    "A6": "F5", "A7": "F7", "A8": "FT7", "A9": "FC5", "A10": "FC3",
    "A11": "FC1", "A12": "C1", "A13": "C3", "A14": "C5", "A15": "T7",
    "A16": "TP7", "A17": "CP5", "A18": "CP3", "A19": "CP1", "A20": "P1",
    "A21": "P3", "A22": "P5", "A23": "P7", "A24": "P9", "A25": "PO7",
    "A26": "PO3", "A27": "O1", "A28": "Iz", "A29": "Oz", "A30": "POz",
    "A31": "Pz", "A32": "CPz",

    # B row
    "B1": "Fpz", "B2": "Fp2", "B3": "AF8", "B4": "AF4", "B5": "AFz",
    "B6": "Fz", "B7": "F2", "B8": "F4", "B9": "F6", "B10": "F8",
    "B11": "FT8", "B12": "FC6", "B13": "FC4", "B14": "FC2", "B15": "FCz",
    "B16": "Cz", "B17": "C2", "B18": "C4", "B19": "C6", "B20": "T8",
    "B21": "TP8", "B22": "CP6", "B23": "CP4", "B24": "CP2", "B25": "P2",
    "B26": "P4", "B27": "P6", "B28": "P8", "B29": "P10", "B30": "PO8",
    "B31": "PO4", "B32": "O2",
}

# Anwenden des offiziellen BioSemi-Mappings
raw_eeg.rename_channels(bios64_to_1020)

# BioSemi-Standardmontage
mont = mne.channels.make_standard_montage("biosemi64")
raw_eeg.set_montage(mont)

# Jetzt können die Sensoren geplottet werden
raw_eeg.plot_sensors(kind='topomap', show_names=True, sphere='auto')


Loading: ..\data\sub-01_P1_raw.fif
Opening raw data file ..\data\sub-01_P1_raw.fif...
Isotrak not found
    Range : 0 ... 7514111 =      0.000 ...  3669.000 secs
Ready.
Reading 0 ... 7514111  =      0.000 ...  3669.000 secs...
✅ Raw data loaded.

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


ValueError: DigMontage is only a subset of info. There are 64 channel positions not present in the DigMontage. The channels missing from the montage are:

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if these are not EEG channels, or use the on_missing parameter if the channel positions are allowed to be unknown in your analyses.

#### Noisy Channels

In [3]:
def visualize_noisy_channels(subj, person="P1"):
    """
    Load the noisy-channel-cleaned raw file and visualize:
    1) EEG traces
    2) Marked bad channels
    3) Topomap of channel variance (works with MNE 1.8 + BioSemi)
    """
    # File from Step 03
    file_path = OUTPUT_DIR / f"sub-{subj}_{person}_raw_noisy_cleaned.fif"
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return

    # Load raw data
    raw = load_raw(OUTPUT_DIR / f"sub-{subj}_{person}_raw.fif", preload=True)

    # Ensure montage exists (BioSemi ActiveTwo 64)
    try:
        raw.get_montage()
        print("Montage found in raw data.")
    except RuntimeError:
        print("Montage not available.")

    # Print info
    print(raw)
    print("Bad channels marked in raw.info['bads']:", raw.info['bads'])

    # --- Plot EEG channels with bad channels highlighted ---
    raw.plot(n_channels=32, block=True,
             title=f"Subject {subj} {person} - Bad Channels Highlighted")

    # ----------------------------------------------------------------------
    # ---------------------- Topomap of Channel Variance -------------------
    # ----------------------------------------------------------------------
    import numpy as np

    picks = mne.pick_types(raw.info, eeg=True)
    data = raw.get_data(picks=picks)
    ch_names = [raw.ch_names[i] for i in picks]

    variances = data.var(axis=1)

    # Get BioSemi layout positions (works without digitizer points)
    layout = mne.find_layout(raw.info)
    pos = layout.pos  # 2D positions of electrodes

    # Create topomap
    im, _ = mne.viz.plot_topomap(
        variances,
        pos,
        cmap="Reds",
        contours=0,
        show=False,
        sensors=False  # we label manually
    )

    # Add channel name labels manually
    for name, (x, y) in zip(ch_names, pos):
        plt.text(x, y, name, fontsize=6, ha='center', va='center')

    plt.colorbar(im)
    plt.title(f"Channel variance - Subject {subj} {person}")
    plt.show()


#### Re-referencing

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / '..' / 'eeg_pipeline').resolve()))

import utils
from utils import load_raw

# --- Original Data ---
raw_orig = load_raw("../data/sub-01_P1_raw.fif")

# --- Re-referenced Data ---
raw_reref = load_raw("../data/sub-01_P1_raw_CAR.fif")

# --- Choose EEG Channels ---
eeg_chs = [ch for ch in raw_orig.ch_names if ch.startswith("1-A") or ch.startswith("1-B")]
raw_orig.pick_channels([c for c in eeg_chs if c == '1-B16'])
raw_reref.pick_channels([c for c in eeg_chs if c == '1-B16'])

# --- Scaling: e.g. ±100 µV, same for all channels ---
scaling = 1000e-6  # Volt

# --- Originalplot ---
raw_orig.plot(n_channels=len(eeg_chs), duration=60, scalings={'eeg': scaling}, title="Original P1")

# --- Re-referenced Plot ---
raw_reref.plot(n_channels=len(eeg_chs), duration=60, scalings={'eeg': scaling}, title="Re-referenced P1")

Opening raw data file ../data/sub-01_P1_raw.fif...


Isotrak not found
    Range : 0 ... 7514111 =      0.000 ...  3669.000 secs
Ready.
Reading 0 ... 7514111  =      0.000 ...  3669.000 secs...
Opening raw data file ../data/sub-01_P1_raw_CAR.fif...


C:\Users\bk57s\Visual Studio Code\EEG_Bala Sharks\eeg_pipeline\utils.py:10: RuntimeWarning: This filename (../data/sub-01_P1_raw_CAR.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(filename, preload=preload)


FileNotFoundError: fname does not exist: "c:\Users\bk57s\Visual Studio Code\EEG_Bala Sharks\sanity_checks\..\data\sub-01_P1_raw_CAR.fif"